# Phase 3B on the SURF A10 VM

Use the frozen selection and original videos copied from Colab. This notebook does not re-screen cases or run Part 1/Phase 1. All outputs and model caches use `/data/yuxuanstorage`. Start with the CPU plan, then VM preflight on both GPUs. Inspect the preflight and relocation-control results before explicitly selecting `full`.


In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

STORAGE_ROOT = Path('/data/yuxuanstorage')
PROJECT_ROOT = STORAGE_ROOT / 'vlm-event-boundary'
PHASE3B_VM_SELECTION = STORAGE_ROOT / 'vlm_phase3b/analysis/selection_v2_controls'
PHASE3B_VM_POOL = STORAGE_ROOT / 'vlm_phase3b/rescue_pool'
PHASE3B_VM_OUTPUT_ROOT = STORAGE_ROOT / 'vlm_phase3b/a10_runs/run_v2_mp'
PHASE3B_VM_GPUS = '0,1'
PHASE3B_VM_EXECUTION_MODE = 'model_parallel'
PHASE3B_VM_WEIGHT_BUDGET_GIB = 10
PHASE3B_VM_STAGE = 'full'  # Requires a completed and reviewed A10 preflight.
PHASE3B_VM_BACKGROUND = True  # Detached tmux job; closing the browser is safe.
PHASE3B_VM_PAUSE_POLICY = 'finished'  # Pause after a verified backup, including failed runs.
PHASE3B_VM_CONFIRM_EXCLUSIVE = True  # Explicit opt-in: Pause stops the whole workspace.
PHASE3B_VM_SURF_CONFIG = STORAGE_ROOT / '.phase3b_private/surf.json'
PHASE3B_VM_BACKUP_DIR = STORAGE_ROOT / 'backups'

if not (PROJECT_ROOT / 'scripts/run_phase3b_vm.py').is_file():
    raise FileNotFoundError('Set PROJECT_ROOT to the repository on persistent storage.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
vm_runner_options = [
    '--selection_dir', str(PHASE3B_VM_SELECTION),
    '--rescue_pool_root', str(PHASE3B_VM_POOL),
    '--gpus', PHASE3B_VM_GPUS,
    '--execution_mode', PHASE3B_VM_EXECUTION_MODE,
    '--gpu_weight_budget_gib', str(PHASE3B_VM_WEIGHT_BUDGET_GIB),
]
vm_common = [*vm_runner_options,
    '--output_root', str(PHASE3B_VM_OUTPUT_ROOT),
    '--storage_root', str(STORAGE_ROOT), '--project_root', str(PROJECT_ROOT),
]
PHASE3B_VM_LIFECYCLE = PHASE3B_VM_OUTPUT_ROOT.with_name(PHASE3B_VM_OUTPUT_ROOT.name + '_lifecycle')


## CPU Plan

Checks all frozen video paths and hashes before loading a model. The `1001_phase3B_output.zip` archive contains selection and mapping files but no videos: copy the Drive rescue pool and the existing L5 full videos separately.


In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/run_phase3b_vm.py', '--stage', 'plan', *vm_common], check=True)


## Run or Resume

One model spans both A10s; shards run sequentially in one worker. A 10 GiB weight-placement budget per GPU reserves memory for unchanged eager attention. The two-case distributed preflight and matched relocation control must pass before `full` is allowed. Independent replicas are optional, but the first A10 run exhausted VRAM during eager attention. Re-running the same configuration resumes its saved checkpoints. Do not reuse the A100 checkpoint directory or the failed independent-worker `run_v1`. Frozen cases and model caches are reused; no re-screening is performed.

Background mode uses `tmux` and returns immediately. Install it in a VM terminal with `sudo apt install tmux`; monitor the printed persistent log or use `tmux attach`. Do not launch a second copy while a direct CLI preflight is still running. Background completion automatically creates and verifies a full backup. This notebook is configured for `full` with Pause policy `finished`: successful and failed runs both request Pause after a verified full backup. Complete the README's hidden-token SURF setup and ensure no other person or job needs the workspace before launching. Backup/verification failures or other active GPU jobs block Pause. No token belongs in a notebook cell.


In [ ]:
if PHASE3B_VM_STAGE not in {'preflight', 'full', 'analyze'}:
    raise ValueError('Choose preflight, full, or analyze.')
if PHASE3B_VM_BACKGROUND:
    launch_options = [
        '--stage', PHASE3B_VM_STAGE, '--session_name', 'phase3b_' + PHASE3B_VM_STAGE,
        '--output_root', str(PHASE3B_VM_OUTPUT_ROOT),
        '--storage_root', str(STORAGE_ROOT), '--project_root', str(PROJECT_ROOT),
        '--backup_dir', str(PHASE3B_VM_BACKUP_DIR),
        '--pause_policy', PHASE3B_VM_PAUSE_POLICY, '--surf_config', str(PHASE3B_VM_SURF_CONFIG),
    ]
    if PHASE3B_VM_CONFIRM_EXCLUSIVE:
        launch_options.append('--confirm_exclusive_workspace')
    subprocess.run([sys.executable, '-u', 'scripts/launch_phase3b_vm.py', *launch_options, '--', *vm_runner_options], check=True)
else:
    if PHASE3B_VM_PAUSE_POLICY != 'off':
        raise ValueError('Automatic Pause requires background mode; never stop the VM before the backup cell.')
    subprocess.run([sys.executable, '-u', 'scripts/run_phase3b_vm.py', '--stage', PHASE3B_VM_STAGE, *vm_common], check=True)


## Back Up After Every Run

In background mode this cell only displays the lifecycle status: it must not package a still-running experiment. The detached wrapper makes and verifies the full backup after its workers stop. In foreground mode this cell creates and verifies the backup directly. Download **all** parts, `backup_manifest.json`, and `SHA256SUMS` through the VM file browser or scp/rsync and verify them on your local computer. Packaging on VM is not yet a local backup. If persistent disk space cannot hold another copy, transfer the run directory and frozen videos directly instead; do not delete checkpoints. After automatic Pause, confirm `paused` in the SURF portal; resume when needed to download.


In [ ]:
from scripts.backup_phase3b import create_backup, verify_backup

if PHASE3B_VM_BACKGROUND:
    status_path = PHASE3B_VM_LIFECYCLE / 'job_status.json'
    if status_path.is_file():
        lifecycle_status = json.loads(status_path.read_text())
        print(json.dumps(lifecycle_status, indent=2))
        if lifecycle_status.get('backup_verified'):
            print('Download every file in:', lifecycle_status['backup_dir'])
        else:
            print('No verified backup yet. Monitor the job log; this cell will not duplicate or interrupt the job.')
    else:
        print('Waiting for the detached wrapper to start. Monitor:', PHASE3B_VM_LIFECYCLE / 'job.log')
else:
    backup_bundle = create_backup(PHASE3B_VM_OUTPUT_ROOT, PHASE3B_VM_BACKUP_DIR, reports_only=False)
    verify_backup(backup_bundle)
    print('Download every file in:', backup_bundle)
    print('Files:', [path.name for path in sorted(backup_bundle.iterdir())])
print('On your local computer, verify: python scripts/backup_phase3b.py --verify <downloaded-directory>')
print('Keep the VM checkpoints until the local backup has been verified.')
